In [2]:
# train.py
import os
import json
from dataclasses import dataclass
from typing import List, Dict, Any
from functools import partial

import numpy as np
# from datasets import load_dataset, Dataset, concatenate_datasets
from torch.utils.data import Dataset 
from transformers import (
    AutoTokenizer,
    AutoModel,
    TrainingArguments,
    DataCollatorForLanguageModeling,
    Trainer,
AutoModelForCausalLM
)
from transformers.trainer_utils import get_last_checkpoint

import torch
from sklearn.model_selection import train_test_split
# from tasks import render_example
import torch.nn as nn
import pandas as pd

import torch

from dataclasses import dataclass
from typing import Dict, Any, Callable, Optional
import evaluate
import shutil
import glob
from datetime import datetime
# from peft import LoraConfig, get_peft_model

import os
import argparse
import numpy as np
import torch
import wandb
import evaluate
from datetime import datetime
from sklearn.model_selection import train_test_split
from transformers import AutoTokenizer, TrainingArguments, Trainer
import sys
from pathlib import Path


/ihome/xli/sek188/MSTHESIS/envs/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
odf = pd.read_csv("ldf_with_entropies_ntext.csv")

fdf = odf.copy()
from sklearn.preprocessing import MinMaxScaler
fdf['H_obs_scaled'] = MinMaxScaler().fit_transform(fdf[['H_obs']])
fdf['H_pred_scaled'] = MinMaxScaler().fit_transform(fdf[['H_pred']])
fdf['H_model'] = (fdf['H_obs_scaled'] - fdf['H_pred_scaled']).clip(lower=0)

In [38]:
fdf.columns

Index(['survey_record_id', 'sentence_id', 'label', 'consensus_label', 'type',
       'topic', 'outlet', 'age', 'gender', 'education',
       'native_english_speaker', 'political_ideology', 'followed_news_outlets',
       'news_check_frequency', 'survey_completed', 'label_binary',
       'biased_count', 'total_count', 'poli_leaning_binned', 'age_binned',
       'education_binned', 'num_followed_outlets', 'emb_pca_1', 'emb_pca_2',
       'emb_pca_3', 'emb_pca_4', 'emb_pca_5', 'custom_idx', 'out_emb1',
       'out_emb2', 'out_emb3', 'p_hat', 'H_obs', 'H_pred', 'H_model', 'text',
       'H_obs_scaled', 'H_pred_scaled'],
      dtype='str')

We have the following models to compare:
- Lora, zerocats, no_ambi
- Lora, zerocats
- Lora, no_ambi
- Lora
- Lora, deberta
- Lora, full spinde similarity


Our goal is that lora > any of the abalations, and lora \geq full spinde

In [4]:
# this tests Lora, zerocats, no_ambi
# TinyLlama-1.1B-Chat-v1.0_H_model_0410_0746
df_l_zc_na = pd.read_csv('./model/TinyLlama-1.1B-Chat-v1.0_H_model_0410_0746/test_preds.csv')
# df_l_zc_na.rename({"pred_bias" : "df_l_zc_na_pred_bias", "pred_ambi" : "df_l_zc_na_pred_ambi", 

In [5]:
# this tests Lora, zerocats
# TinyLlama-1.1B-Chat-v1.0_H_model_0410_0704
df_l_zc_a = pd.read_csv('./model/TinyLlama-1.1B-Chat-v1.0_H_model_0410_0704/test_preds.csv')

In [6]:
# this tests Lora, no_ambi
# TinyLlama-1.1B-Chat-v1.0_H_model_0410_0946
df_l_c_na = pd.read_csv('./model/TinyLlama-1.1B-Chat-v1.0_H_model_0410_0946/test_preds.csv')

In [7]:
# this tests Lora,
# TinyLlama-1.1B-Chat-v1.0_H_model_0409_1920
df_l_c_a = pd.read_csv('./model/TinyLlama-1.1B-Chat-v1.0_H_model_0409_1920/test_preds.csv')

In [8]:
# this tests Lora,deberta
# deberta-v3-large_H_model_0414_1233
df_l_c_a_db = pd.read_csv('./model/deberta-v3-large_H_model_0414_1233/test_preds.csv')

In [9]:
# this tests Lora,deberta
# xlm-roberta-base_H_model_0414_1254
df_l_c_a_rb = pd.read_csv('./model/xlm-roberta-base_H_model_0414_1254/test_preds.csv')

In [34]:
model_dfs = {
    'l_c_a_db': df_l_c_a_db,
    'l_c_a_rb': df_l_c_a_rb,
    'l_c_a': df_l_c_a,
    'df_l_c_na': df_l_c_na,
    'l_zc_a': df_l_zc_a,
    'l_zc_na': df_l_zc_na
    
}

combined = None
for model_name, df in model_dfs.items():
    deduped = df.groupby('text').agg(
        label_binary=('label_binary', 'first'),
        pred_bias=('pred_bias', lambda x: x.mode()[0]),
        pred_ambi=('pred_ambi', 'mean'),
        pred_variance=('pred_ambi', 'var')
    ).reset_index()
    
    if combined is None:
        combined = deduped[['text', 'label_binary']].copy()
    
    combined = combined.merge(
        deduped[['text', 'pred_bias', 'pred_ambi', 'pred_variance']].rename(columns={
            'pred_bias': f'pred_bias_{model_name}',
            'pred_ambi': f'pred_ambi_{model_name}',
            'pred_variance': f'pred_variance_{model_name}',
        }),
        on='text'
    )

combined = combined.merge(fdf[['text', 'consensus_label', 'H_model', 'H_pred_scaled']].drop_duplicates('text'), on='text')

# # When you need long format for seaborn/groupby:
# long = combined.melt(
#     id_vars=['text', 'label_binary'],
#     var_name='col', value_name='value'
# )
# long[['metric', 'model']] = long['col'].str.extract(r'(pred_\w+)_(.+)')

In [22]:
combined.columns

Index(['text', 'label_binary', 'pred_bias_l_c_a_db', 'pred_ambi_l_c_a_db',
       'pred_variance_l_c_a_db', 'pred_bias_l_c_a_rb', 'pred_ambi_l_c_a_rb',
       'pred_variance_l_c_a_rb', 'pred_bias_l_c_a', 'pred_ambi_l_c_a',
       'pred_variance_l_c_a', 'pred_bias_df_l_c_na', 'pred_ambi_df_l_c_na',
       'pred_variance_df_l_c_na', 'pred_bias_l_zc_a', 'pred_ambi_l_zc_a',
       'pred_variance_l_zc_a', 'pred_bias_l_zc_na', 'pred_ambi_l_zc_na',
       'pred_variance_l_zc_na', 'consensus_label'],
      dtype='str')

In [23]:
# df_l_c_a_db.shape
# 0.2 * fdf.shape[0]
# df_l_c_a_db['text'].nunique()
combined['consensus_label'].value_counts()

consensus_label
Biased          442
Non-biased      332
No agreement     90
Name: count, dtype: int64

In [24]:
no_agree = combined[combined['consensus_label'].str.strip() == 'No agreement']
print(f"No agreement sentences: {len(no_agree)}")

bias_cols = [c for c in combined.columns if c.startswith('pred_bias')]

# what does each model predict on no-agreement sentences
print("\nPrediction distributions on No Agreement:")
print(no_agree[bias_cols].apply(lambda x: x.value_counts()).T)

# vs the full set
print("\nPrediction distributions overall:")
print(combined[bias_cols].apply(lambda x: x.value_counts()).T)

# error rate by consensus label for each model
for col in bias_cols:
    combined['error'] = (combined['label_binary'] != combined[col]).astype(int)
    print(f"\n{col} error rate by consensus:")
    print(combined.groupby('consensus_label')['error'].mean())

No agreement sentences: 90

Prediction distributions on No Agreement:
                        0     1
pred_bias_l_c_a_db    NaN  90.0
pred_bias_l_c_a_rb    9.0  81.0
pred_bias_l_c_a      53.0  37.0
pred_bias_df_l_c_na  49.0  41.0
pred_bias_l_zc_a     44.0  46.0
pred_bias_l_zc_na    44.0  46.0

Prediction distributions overall:
                        1    0
pred_bias_l_c_a_db   1542    2
pred_bias_l_c_a_rb   1394  150
pred_bias_l_c_a       970  574
pred_bias_df_l_c_na   985  559
pred_bias_l_zc_a     1063  481
pred_bias_l_zc_na     981  563

pred_bias_l_c_a_db error rate by consensus:
consensus_label
Biased          0.210407
No agreement    0.488889
Non-biased      0.728916
Name: error, dtype: float64

pred_bias_l_c_a_rb error rate by consensus:
consensus_label
Biased          0.248869
No agreement    0.455556
Non-biased      0.626506
Name: error, dtype: float64

pred_bias_l_c_a error rate by consensus:
consensus_label
Biased          0.276018
No agreement    0.611111
Non-biased      0.

In [26]:
# no_agree['label_binary'].value_counts()
print(combined.groupby('consensus_label')['pred_ambi_l_c_a'].mean())

consensus_label
Biased          0.135733
No agreement    0.142323
Non-biased      0.147895
Name: pred_ambi_l_c_a, dtype: float64


In [29]:
# merge gold ambi back in
print(combined[['pred_ambi_l_c_a', 'H_model']].corr())

                 pred_ambi_l_c_a   H_model
pred_ambi_l_c_a         1.000000  0.045116
H_model                 0.045116  1.000000


In [30]:
print(combined['H_model'].describe())
print(combined['pred_ambi_l_c_a'].describe())

count    1544.000000
mean        0.147431
std         0.114054
min         0.000000
25%         0.030138
50%         0.150603
75%         0.231291
max         0.618220
Name: H_model, dtype: float64
count    1544.000000
mean        0.135929
std         0.032014
min         0.075883
25%         0.110761
50%         0.136308
75%         0.148164
max         0.250535
Name: pred_ambi_l_c_a, dtype: float64


In [32]:
# quick validation of claim 1
# combined['h_bin'] = pd.qcut(combined['H_model'], 5, labels=False)
combined['h_bin'] = pd.qcut(combined['H_model'], 5, labels=False, duplicates='drop')
error_cols = [f'pred_bias_{m}' for m in model_dfs.keys()]
for col in error_cols:
    combined['err'] = (combined['label_binary'] != combined[col]).astype(int)
    
    print(col, combined.groupby('h_bin')['err'].mean().values)
    
    

pred_bias_l_c_a_db [0.36084142 0.45454545 0.4368932  0.36569579]
pred_bias_l_c_a_rb [0.34627832 0.42857143 0.43365696 0.34627832]
pred_bias_l_c_a [0.2605178  0.40909091 0.46278317 0.3592233 ]
pred_bias_df_l_c_na [0.27022654 0.37987013 0.45954693 0.39482201]
pred_bias_l_zc_a [0.20550162 0.33766234 0.44983819 0.40776699]
pred_bias_l_zc_na [0.22168285 0.35714286 0.48543689 0.37216828]


In [35]:
# quick validation of claim 1
# combined['h_bin'] = pd.qcut(combined['H_model'], 5, labels=False)
combined['p_bin'] = pd.qcut(combined['H_pred_scaled'], 5, labels=False, duplicates='drop')
error_cols = [f'pred_bias_{m}' for m in model_dfs.keys()]
for col in error_cols:
    combined['err'] = (combined['label_binary'] != combined[col]).astype(int)
    
    print(col, combined.groupby('p_bin')['err'].mean().values)

pred_bias_l_c_a_db [0.13592233 0.39158576 0.43506494 0.50485437 0.51132686]
pred_bias_l_c_a_rb [0.14239482 0.38187702 0.41558442 0.47249191 0.48867314]
pred_bias_l_c_a [0.06796117 0.26860841 0.3961039  0.51456311 0.50485437]
pred_bias_df_l_c_na [0.07443366 0.30420712 0.39935065 0.53398058 0.46278317]
pred_bias_l_zc_a [0.05825243 0.25889968 0.36363636 0.4789644  0.44660194]
pred_bias_l_zc_na [0.06148867 0.27831715 0.36363636 0.50809061 0.44660194]


In [37]:
hi_obs = combined['H_pred_scaled'] > combined['H_pred_scaled'].median()  # using h_pred as H_obs proxy
hi_model = combined['H_model'] > combined['H_model'].median()

print("Case 3 - polarized:")
print(combined[hi_obs & ~hi_model]['text'].sample(5).values)

print("\nCase 4 - ambivalent:")
print(combined[hi_obs & hi_model]['text'].sample(5).values)

Case 3 - polarized:
<ArrowStringArray>
[         'Cuban slave doctors that have escaped the system have said that, while working abroad, the Castro regime forced them to invent patients and throw away perfectly good medicine to boost the statistics on the number of patients Cuban doctors actually treat.',
                                                'Colin P. Clarke has been teaching a course on terrorism and insurgency at Carnegie Mellon University in Pittsburgh for four years, and much more of his class these days is devoted to white supremacy than in the past.',
                                                                                                                                               'A solid majority of student loan borrowers in a recent survey said they would trade their 2020 vote for debt forgiveness.',
                                                             'Rep. Ilhan Omar of Minnesota recently sparked a major controversy when she criticized U.S. supp

In [39]:
sdf = fdf.groupby('text').agg(
    biased_count=('biased_count', 'first'),
    total_count=('total_count', 'first'),
    consensus_label=('consensus_label', 'first'),
    H_obs=('H_obs', 'first'),
    H_pred=('H_pred', 'first'),
).reset_index()

sdf['majority_count'] = sdf[['biased_count', 'total_count']].apply(
    lambda r: max(r['biased_count'], r['total_count'] - r['biased_count']), axis=1
)
sdf['majority_ratio'] = sdf['majority_count'] / sdf['total_count']

print(sdf['majority_ratio'].describe())
print(sdf.groupby('consensus_label')['majority_ratio'].describe())

count    1700.000000
mean        0.736094
std         0.147275
min         0.500000
25%         0.600000
50%         0.727273
75%         0.833333
max         1.000000
Name: majority_ratio, dtype: float64
                 count      mean       std       min       25%  50%  75%  \
consensus_label                                                            
Biased           487.0  0.767773  0.134051  0.545455  0.651515  0.8  0.9   
No agreement      94.0  0.500484  0.004688  0.500000  0.500000  0.5  0.5   
Non-biased       369.0  0.725116  0.115915  0.545455  0.636364  0.7  0.8   

                      max  
consensus_label            
Biased           1.000000  
No agreement     0.545455  
Non-biased       1.000000  
